# Notebook 1 — FLAN-T5-base: Generic Prompt Baseline

**Model:** `google/flan-t5-base` (~250M parameters)  
**Task:** Text simplification using a generic, non-domain-specific prompt  
**Input:** `data/final/dataset_annotation_v3.csv`  
**Output column:** `output_generic`  
**Saved to:** `data/final/dataset_model_v2.csv`

---
This is the **baseline** run. The prompt is intentionally simple and not tailored to the financial domain.

## 0. Environment Info

In [ ]:
# Environment info — run this first to log versions for reproducibility
import platform, importlib
for pkg in ["torch", "transformers", "sentencepiece"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<15}: {mod.__version__}")
    except Exception:
        print(f"{pkg:<15}: not installed")
print(f"{'python':<15}: {platform.python_version()}")
print(f"{'platform':<15}: {platform.platform()}")

## 1. Install Dependencies

In [ ]:
import subprocess, sys

packages = ['transformers==4.40.0', 'torch==2.2.0', 'sentencepiece==0.1.99']
for pkg in packages:
    base = pkg.split('==')[0].replace('-', '_')
    try:
        __import__(base)
        print(f'  {pkg} already installed')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  {pkg} installed')

## 2. Imports

In [ ]:
import pandas as pd
import torch
import re
from transformers import T5ForConditionalGeneration, T5Tokenizer

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device    : {DEVICE}")

## 2b. Set Random Seed

In [ ]:
import torch, random, numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Random seed set to {SEED}")

## 3. Load Dataset

In [ ]:
INPUT_PATH  = "../data/final/dataset_annotation_v3.csv"
OUTPUT_PATH = "../data/final/dataset_model_v2.csv"

df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
df[["candidate_id", "text"]].head(3)

## 4. Load FLAN-T5-base Model

In [ ]:
model_name = "google/flan-t5-base"

print(f"Loading tokenizer: {model_name}")
tokenizer = T5Tokenizer.from_pretrained(model_name)

print(f"Loading model: {model_name}")
model = T5ForConditionalGeneration.from_pretrained(model_name)
model = model.to(DEVICE)
model.eval()
print("Model loaded and ready.")

## 5. Define Prompt

> **Generic prompt** — no domain context, just a plain simplification instruction.

In [ ]:
PROMPT_PREFIX = "Simplify the following text so that a non-expert can understand it:\n\n"

# Preview prompt with first row
example_prompt = PROMPT_PREFIX + str(df["text"].iloc[0])
print("=== Example Prompt ===")
print(example_prompt)
print(f"\nToken count: {len(tokenizer(example_prompt)['input_ids'])}")

## 6. Run Inference

In [ ]:
outputs = []
total = len(df)

for i, row in df.iterrows():
    prompt = PROMPT_PREFIX + str(row["text"])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    outputs.append(decoded)

    row_num = i + 1
    if row_num % 10 == 0:
        print(f"Processed {row_num}/{total} rows...")

print(f"Processed {total}/{total} rows. Done.")

## 7. Save Results

In [ ]:
df["output_generic"] = outputs
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(f"Columns: {df.columns.tolist()}")

## 8. Summary Statistics

In [ ]:
def approx_fkgl(text):
    words = str(text).split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', str(text)) if s.strip()]
    num_sentences = max(len(sentences), 1)
    num_words = max(len(words), 1)
    num_syllables = sum(max(1, len(w) // 3) for w in words)
    return round(0.39 * (num_words / num_sentences) + 11.8 * (num_syllables / num_words) - 15.59, 2)

df["wc_generic"]   = df["output_generic"].apply(lambda x: len(str(x).split()))
df["fkgl_generic"] = df["output_generic"].apply(approx_fkgl)
identical = (df["output_generic"].str.strip() == df["text"].str.strip()).sum()

print("=" * 45)
print("SUMMARY — FLAN-T5-base Generic Prompt")
print("=" * 45)
print(f"Total rows processed        : {len(df)}")
print(f"Avg word count (output)     : {df['wc_generic'].mean():.2f}")
print(f"Avg FKGL (output)           : {df['fkgl_generic'].mean():.2f}")
print(f"Rows identical to input     : {identical}/{len(df)}")
print()
print(f"[Reference] Avg word count (original) : {df['text'].apply(lambda x: len(str(x).split())).mean():.2f}")
print(f"[Reference] Avg FKGL (original)       : {df['text'].apply(approx_fkgl).mean():.2f}")

## 9. Sample Comparison

In [ ]:
# Show 5 rows where the model actually changed the text
changed = df[df["output_generic"].str.strip() != df["text"].str.strip()].head(5)

print(f"Rows with actual changes: {(df['output_generic'].str.strip() != df['text'].str.strip()).sum()}/100\n")
for _, row in changed.iterrows():
    print(f"[Row {row['candidate_id']}]")
    print(f"  ORIGINAL : {row['text'][:120]}")
    print(f"  OUTPUT   : {row['output_generic'][:120]}")
    print()

## 10. Token Length Distribution

How many rows were likely truncated at 512 tokens?

In [ ]:
df["token_count"] = df["text"].apply(
    lambda t: len(tokenizer(PROMPT_PREFIX + str(t))["input_ids"])
)

truncated = (df["token_count"] > 512).sum()
print(f"Rows exceeding 512 tokens (truncated): {truncated}/{len(df)}")
print()
print(df["token_count"].describe().round(2))

---
## Notes

- **FLAN-T5-base** is a 250M parameter encoder-decoder model, designed for instruction-following tasks (QA, translation, summarisation). It is **not** trained specifically for text simplification.
- The generic prompt gives it minimal guidance, making this the weakest baseline.
- Rows with output identical to input indicate the model chose to copy rather than rephrase — common when the prompt is vague or the input exceeds the 512-token window.
- See `notebook_02` for the financial domain-specific prompt on the same model.